# Integration scSHM data with WILLOW

In [1]:
#Load packages
options(warn=-1)
library("IRdisplay")
library(tidyverse)
library(ggplot2)

── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.5     ✔ purrr   0.3.4
✔ tibble  3.1.6     ✔ dplyr   1.0.8
✔ tidyr   1.2.0     ✔ stringr 1.4.0
✔ readr   2.1.2     ✔ forcats 0.5.1

── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()



In [2]:
df_summary <- read.csv("output/df_summary.csv")

In [3]:
print("Number of scSHM cells")
df_summary %>% pull(cell) %>% unique() %>% length()

# scSHM cells list
scSHM_cells <-df_summary %>% pull(cell) %>% unique()

[1] "Number of scSHM cells"


[1] 71

In [19]:
clone_id <- readRDS("~/repositories/FL_10X_2/250_Clonotypes.freeze02/outs/clonesB_20210823.rds") %>% mutate(cell = gsub(".{2}$", "", barcode)) %>%
            filter(cell %in% scSHM_cells)

In [5]:
#### is_cell : True or False value indicating whether the barcode was called as a cell.

#In addition to the three requirements listed above, Cell Ranger v3.1 (and later) has a filter to account for noise introduced by plasma cells and B cells 
#containing large amounts of RNA (as documented in the Cell Ranger 3.1 release notes). This 1) tightens the is_cell filter for low-frequency clones that 
#share a chain with a higher-frequency or large clone, and 2) shrinks high-frequency clones to remove noise from mRNA leakage caused by sample processing 
#(e.g. not due to biological clonal expansion).

#https://support.10xgenomics.com/single-cell-vdj/software/pipelines/latest/algorithms/cell-calling

#Compute the N50 value of the number of read pairs per UMI, across all barcodes. For a given barcode, if the maximum read pair count across filtered UMIs 
#is less than 3% of this N50, do not call the barcode a cell. This provides some protection against transcripts arising from index hopping on an Illumina 
#flowcell, and from other forms of cross-library contamination.

#maybe there are not so much reads for the UMI by problems in the aligment??? for that reason is considered as FALSE

In [6]:
str(clone_id)

Classes ‘data.table’ and 'data.frame':	165 obs. of  55 variables:
 $ source                : chr  "K1B" "K1B" "K1B" "K1B" ...
 $ cell                  : chr  "AACTCAGCATTCGACA" "AACTCAGCATTCGACA" "AACTCAGCATTCGACA" "AACTCTTGTGCAACTT" ...
 $ Sequence ID           : chr  "AACTCAGCATTCGACA-1_contig_1" "AACTCAGCATTCGACA-1_contig_2" "AACTCAGCATTCGACA-1_contig_3" "AACTCTTGTGCAACTT-1_contig_1" ...
 $ V-DOMAIN Functionality: chr  "productive" "productive" "productive" "productive" ...
 $ V-GENE and allele     : chr  "IGKV1-17*01 F" "IGKV1-17*01 F" "IGHV3-30*03 F, or IGHV3-30*18 F or IGHV3-30-5*01 F" "IGKV3-20*01 F" ...
 $ J-GENE and allele     : chr  "IGKJ4*01 F" "IGKJ4*01 F" "IGHJ6*02 F" "IGKJ2*01 F" ...
 $ D-GENE and allele     : chr  NA NA "IGHD3-10*01 F" NA ...
 $ V-REGION              : chr  "gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccttcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcag"| __truncated__ "gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccatcacttgcc

In [13]:
options(repr.matrix.max.cols = 150)
head(clone_id)

source,cell,Sequence ID,V-DOMAIN Functionality,V-GENE and allele,J-GENE and allele,D-GENE and allele,V-REGION,FR3-IMGT,FR4-IMGT,JUNCTION,Vregion3Prime,FR1-IMGT start,FR3-IMGT end,FR4-IMGT start,J-REGION,CDR3-IMGT,FR4-IMGT end,seq,gene,genes,contigCount,barcode,is_cell,high_confidence,length,chain,v_gene,d_gene,j_gene,c_gene,full_length,productive,cdr3,reads,umis,V-REGION identity %,seqRaw,subject,status,isSinglet,isDoublet,molecule,receptor,v_geneIMGT,isMedSeqLen,distDomCDR3,distDomSeq,domVgeneCR,domVgeneIMGT,clone,count,distVgene,distJgene,distPO
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<int>,<chr>,<lgl>,<lgl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
K1B,AACTCAGCATTCGACA,AACTCAGCATTCGACA-1_contig_1,productive,IGKV1-17*01 F,IGKJ4*01 F,NA,gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccttcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttacttctgtctacagcataatacttaccc,agtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttacttctgt,ttcggcggagggaccaaggtggagatcaaac,tgtctacagcataatacttacccactcgctttc,tgtctacagcataatacttaccc,99,362,390,ctcgctttcggcggagggaccaaggtggagatcaaac,ctacagcataatacttacccactcgct,420,gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccttcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttacttctgtctacagcataatacttacccactcgctttcggcggagggaccaaggtggagatcaaac,K,NA,1,AACTCAGCATTCGACA-1,FALSE,FALSE,556,IGK,IGKV1-17,None,IGKJ4,IGKC,TRUE,TRUE,CLQHNTYPLAF,3644,17,97.13,tggggaggaatcagtcccactcaggacacagcatggacatgagggtccccgctcagctcctggggctcctgctgctctggttcccaggtgccaggtgtgacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccttcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttacttctgtctacagcataatacttacccactcgctttcggcggagggaccaaggtggagatcaaacgaactgtggctgcaccatctgtcttcatcttcccgccatctgatgagcagttgaaatctggaactgcctctgttgtgtgcctgctgaataacttctatcccagagaggccaaagtacagtggaaggtggataacgc,S144,singlet,TRUE,FALSE,nt,BCR,1-17,TRUE,0,3,TRUE,TRUE,NA,NA,NA,NA,NA
K1B,AACTCAGCATTCGACA,AACTCAGCATTCGACA-1_contig_2,productive,IGKV1-17*01 F,IGKJ4*01 F,NA,gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccatcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttatttctgtctacagcataatacttaccc,agtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttatttctgt,ttcggcggagggaccaaggtggagatcaaac,tgtctacagcataatacttacccactcgctttc,tgtctacagcataatacttaccc,99,362,390,ctcgctttcggcggagggaccaaggtggagatcaaac,ctacagcataatacttacccactcgct,420,gacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccatcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatcagcagcctgcagcctgaagattttgcaacttatttctgtctacagcataatacttacccactcgctttcggcggagggaccaaggtggagatcaaac,K,NA,1,AACTCAGCATTCGACA-1,FALSE,FALSE,556,IGK,IGKV1-17,None,IGKJ4,IGKC,TRUE,TRUE,CLQHNTYPLAF,3140,15,97.85,tggggaggaatcagtcccactcaggacacagcatggacatgagggtccccgctcagctcctggggctcctgctgctctggttcccaggtgccaggtgtgacatccagatgacccagtctccatcctccctgtctgcatctgtaggagacagagtcaccatcacttgccgggcaagtcagggcattagaaatgatttaggctggtttcagcagaaaccagggaaagcccctaagcgcctgatctatgctgcatccagtttgcaaagtggggtcccatcaaggttcagcggctatggatctgggacagagttcactctcacaatc

In [14]:
clone_id %>% pull(source) %>% unique()

[1] "K1B" "K2B" "K3B"

In [8]:
clone_id %>% select(cell,chain,"Sequence ID",is_cell,productive, isDoublet,clone) %>% group_by(cell,chain,clone) %>% count() %>%
left_join(df_summary, "cell")

cell,chain,clone,n,subject,vgene_position_aligned,nucl_po,context_po,subregion,variation,umis,productive
<chr>,<chr>,<dbl>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>
AAATGCCTCTGAAAGA,IGH,NA,3,K2B_S13530-LC,10,C,AG.TG,FR1,C-T,16-22,TRUE
AAATGCCTCTGAAAGA,IGL,NA,1,K2B_S13530-LC,10,C,AG.TG,FR1,C-T,16-22,TRUE
AAATGCCTCTGAAAGA,Multi,NA,2,K2B_S13530-LC,10,C,AG.TG,FR1,C-T,16-22,TRUE
AACACGTAGCCACCTG,IGK,13,1,K2B_S13553-LC,165,C,TA.TG,FR2,C-T,5-5,TRUE
AACTCAGCAAATCCGT,IGH,13,1,K2B_S13530-HC,1,G,.AG,FR1,A-G,9-13,TRUE
AACTCAGCAAATCCGT,IGH,13,1,K2B_S13530-HC,13,G,TG.TG,FR1,A-G,10-9,TRUE
AACTCAGCAAATCCGT,Multi,1,1,K2B_S13530-HC,1,G,.AG,FR1,A-G,9-13,TRUE
AACTCAGCAAATCCGT,Multi,1,1,K2B_S13530-HC,13,G,TG.TG,FR1,A-G,10-9,TRUE
AACTCAGCAAATCCGT,Multi,2,1,K2B_S13530-HC,1,G,.AG,FR1,A-G,9-13,TRUE


In [10]:
clone_id %>% select(cell,chain,"Sequence ID",is_cell,productive, isDoublet,clone) %>% filter(is_cell == FALSE ) %>% pull(cell) %>% unique() %>%
length()

[1] 14